# DL Project 1, v5: v4 recipe (s4 + TrivialAugment + SAM rho 0.05), 2 new seeds on all 50k
Each seed is a separate model under 5M params, not an ensemble.

In [ ]:
%%writefile train.py
"""DL Project 1 (Kaggle): a ResNet under 5M trainable parameters for CIFAR-10. Trains, validates, writes a submission CSV."""
import argparse
import csv
import glob
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
_kaggle = sorted(glob.glob("/kaggle/input/**/train_labels.csv", recursive=True))
DATA = os.path.dirname(_kaggle[0]) if _kaggle else os.path.join(HERE, "data.noindex")  # Kaggle notebook or laptop
RUNS = "/kaggle/working/runs" if os.path.isdir("/kaggle/working") else os.path.join(HERE, "runs")
CACHE = "/kaggle/working" if os.path.isdir("/kaggle/working") else DATA                # Kaggle input is read-only
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


# ---------------- data ----------------
def load(split):
    """Decode the PNGs once, then reuse a uint8 cache: x (N,32,32,3), y (N,), ids (N,)."""
    cache = os.path.join(CACHE, f"{split}.npz")
    if not os.path.exists(cache):
        from PIL import Image
        rows = list(csv.DictReader(open(os.path.join(DATA, "train_labels.csv" if split == "train" else "test.csv"))))
        x = np.stack([np.asarray(Image.open(os.path.join(DATA, f"{split}_images", r["filename"])).convert("RGB")) for r in rows])
        np.savez(cache, x=x, y=np.array([int(r.get("label", -1)) for r in rows]), ids=np.array([int(r["id"]) for r in rows]))
    z = np.load(cache)
    return z["x"], z["y"], z["ids"]


def augment(x, pad=4, cutout=16):
    """Random crop (reflect pad), horizontal flip and cutout, done on the GPU for a whole batch."""
    B, C, H, W = x.shape
    d = x.device
    if pad:
        xp = F.pad(x, (pad,) * 4, mode="reflect")
        iy = torch.randint(0, 2 * pad + 1, (B, 1), device=d) + torch.arange(H, device=d)
        ix = torch.randint(0, 2 * pad + 1, (B, 1), device=d) + torch.arange(W, device=d)
        x = xp[torch.arange(B, device=d)[:, None, None, None], torch.arange(C, device=d)[None, :, None, None],
               iy[:, None, :, None], ix[:, None, None, :]]
    x = torch.where(torch.rand(B, 1, 1, 1, device=d) < 0.5, x.flip(3), x)
    if cutout:
        cy = torch.randint(0, H, (B, 1, 1), device=d)
        cx = torch.randint(0, W, (B, 1, 1), device=d)
        ys = torch.arange(H, device=d)[None, :, None]
        xs = torch.arange(W, device=d)[None, None, :]
        hole = ((ys - cy).abs() < cutout // 2) & ((xs - cx).abs() < cutout // 2)
        x = x * (~hole)[:, None].to(x.dtype)
    return x


def trivial_aug(x):
    """TrivialAugment-style: each image gets ONE random op at a random strength. x is a [0,1] batch on the GPU.
    Ops: identity, brightness, contrast, saturation, sharpness, posterize, solarize, autocontrast,
    rotate, shear x/y, translate x/y. Every op is computed for the whole batch, then each image keeps its own."""
    B, d = x.shape[0], x.device
    op = torch.randint(0, 13, (B,), device=d)
    m = torch.rand(B, device=d)                                                   # strength in [0, 1]
    sgn = torch.where(torch.rand(B, device=d) < 0.5, -1.0, 1.0)
    f = (1 + 0.9 * m * sgn).view(B, 1, 1, 1)                                      # blend factor in [0.1, 1.9]
    v = lambda t: t.view(B, 1, 1, 1)
    gray = (0.299 * x[:, :1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:])
    blur = F.conv2d(F.pad(x, (1, 1, 1, 1), mode="replicate"),
                    torch.tensor([[1., 1, 1], [1, 5, 1], [1, 1, 1]], device=d).div(13).expand(3, 1, 3, 3), groups=3)
    q = v(2 ** torch.floor(m * 5))                                                # posterize: drop 0..4 bits
    lo, hi = x.amin((2, 3), keepdim=True), x.amax((2, 3), keepdim=True)
    cand = [x, x * f, v(gray.mean((1, 2, 3))) + f * (x - v(gray.mean((1, 2, 3)))), gray + f * (x - gray),
            blur + f * (x - blur), torch.floor((x * 255 + 1e-3) / q) * q / 255, torch.where(x >= v(1 - m), 1 - x, x),
            (x - lo) / (hi - lo).clamp_min(1e-3)]
    out = torch.zeros_like(x)
    for k, c in enumerate(cand):
        out = torch.where(v(op == k), c, out)
    # geometric ops: one affine warp per image (identity for the colour ops)
    theta = torch.zeros(B, 2, 3, device=d)
    theta[:, 0, 0] = theta[:, 1, 1] = 1
    s = sgn * m
    ang = s * math.radians(30)
    r = op == 8
    theta[r, 0, 0], theta[r, 0, 1], theta[r, 1, 0], theta[r, 1, 1] = ang[r].cos(), -ang[r].sin(), ang[r].sin(), ang[r].cos()
    theta[op == 9, 0, 1] = 0.3 * s[op == 9]                                       # shear x
    theta[op == 10, 1, 0] = 0.3 * s[op == 10]                                     # shear y
    theta[op == 11, 0, 2] = 0.5 * s[op == 11]                                     # translate x, up to 8 px
    theta[op == 12, 1, 2] = 0.5 * s[op == 12]                                     # translate y, up to 8 px
    geo = op >= 8
    if geo.any():
        grid = F.affine_grid(theta[geo], list(x[geo].shape), align_corners=False)
        out[geo] = F.grid_sample(x[geo], grid, padding_mode="zeros", align_corners=False)
    return out.clamp(0, 1)


# ---------------- model ----------------
class BasicBlock(nn.Module):
    """y = ReLU(S(x) + F(x)). F = conv-BN-ReLU-conv-BN. S = identity, or a 1x1 conv-BN projection when the shape changes."""
    def __init__(self, ci, co, stride):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(ci, co, 3, stride, 1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                               nn.Conv2d(co, co, 3, 1, 1, bias=False), nn.BatchNorm2d(co))
        self.s = nn.Identity() if stride == 1 and ci == co else \
            nn.Sequential(nn.Conv2d(ci, co, 1, stride, bias=False), nn.BatchNorm2d(co))

    def forward(self, x):
        return F.relu(self.s(x) + self.f(x))


class ResNet(nn.Module):
    """3x3 stem, then stages of BasicBlocks (stride 2 at the start of every stage after the first), global avg pool, linear."""
    def __init__(self, widths, blocks, p_drop=0.0, n_cls=10):
        super().__init__()
        layers = [nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True)]
        ci = widths[0]
        for i, (w, n) in enumerate(zip(widths, blocks)):
            for j in range(n):
                layers.append(BasicBlock(ci, w, 2 if (i > 0 and j == 0) else 1))
                ci = w
        self.features = nn.Sequential(*layers)
        self.drop = nn.Dropout(p_drop)
        self.fc = nn.Linear(ci, n_cls)

    def forward(self, x):
        return self.fc(self.drop(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1)))


ARCHS = {
    "r18n": dict(widths=[40, 80, 160, 320], blocks=[2, 2, 2, 2]),   # ResNet-18 layout, narrow
    "s4": dict(widths=[64, 128, 256, 320], blocks=[2, 2, 2, 1]),    # wide early stages, short last stage
    "s3": dict(widths=[64, 128, 256], blocks=[3, 3, 3]),            # 3 stages, deeper, no 4x4 stage
    "s4w": dict(widths=[72, 144, 256, 336], blocks=[2, 2, 2, 1]),   # s4 widened to use the budget (4.88M)
    "r18w": dict(widths=[96, 192, 384, 768], blocks=[2, 2, 2, 2]),  # TEACHER only (25M params, no budget)
}


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# ---------------- train ----------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", default="s4", choices=ARCHS)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--bs", type=int, default=256)
    ap.add_argument("--lr", type=float, default=0.2)
    ap.add_argument("--wd", type=float, default=5e-4)
    ap.add_argument("--ls", type=float, default=0.1, help="label smoothing")
    ap.add_argument("--cutout", type=int, default=16)
    ap.add_argument("--drop", type=float, default=0.0)
    ap.add_argument("--ta", action="store_true", help="TrivialAugment-style colour/geometry ops")
    ap.add_argument("--cutmix", type=float, default=0.0, help="probability of CutMix per batch")
    ap.add_argument("--teacher", default="", help="knowledge distillation: arch:path.pt[,arch:path.pt] (ensemble)")
    ap.add_argument("--kd_a", type=float, default=0.9, help="weight of the distillation loss")
    ap.add_argument("--kd_t", type=float, default=4.0, help="distillation temperature")
    ap.add_argument("--sam", type=float, default=0.0, help="SAM radius rho (0 = plain SGD); costs 2 passes per step")
    ap.add_argument("--tta", type=int, default=2, choices=[2, 10], help="test views: flip, or flip x 5 shifts")
    ap.add_argument("--warmup", type=float, default=0.05, help="fraction of steps")
    ap.add_argument("--full", action="store_true", help="train on all 50k (no validation split)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--name", default=None)
    ap.add_argument("--bench", action="store_true", help="time a few steps of every arch and exit")
    ap.add_argument("--limit", type=int, default=0, help="smoke test: use only this many training images")
    ap.add_argument("--device", default=None)
    a = ap.parse_args()
    global DEV
    DEV = a.device or DEV
    torch.manual_seed(a.seed)
    torch.backends.cudnn.benchmark = True
    amp = DEV in ("cuda", "mps")
    scaler = torch.amp.GradScaler(DEV, enabled=amp)

    if a.bench:
        for k, cfg in ARCHS.items():
            m = ResNet(**cfg).to(DEV)
            opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9)
            x = torch.randn(a.bs, 3, 32, 32, device=DEV); y = torch.randint(0, 10, (a.bs,), device=DEV)
            for i in range(13):
                if i == 3:
                    torch.mps.synchronize() if DEV == "mps" else torch.cuda.synchronize() if DEV == "cuda" else None
                    t = time.time()
                with torch.autocast(DEV, dtype=torch.float16, enabled=amp):
                    loss = F.cross_entropy(m(augment(x)), y)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            torch.mps.synchronize() if DEV == "mps" else torch.cuda.synchronize() if DEV == "cuda" else None
            ips = 10 * a.bs / (time.time() - t)
            print(f"{k:5s} params {n_params(m):,}  {ips:,.0f} img/s  ~{50000 / ips:.0f} s/epoch")
        return

    tags = ("_ta" if a.ta else "") + (f"_cm{a.cutmix:g}" if a.cutmix else "") + ("_kd" if a.teacher else "") + \
        (f"_sam{a.sam:g}" if a.sam else "")
    name = a.name or f"{a.arch}{tags}_e{a.epochs}{'_full' if a.full else ''}_s{a.seed}"
    os.makedirs(RUNS, exist_ok=True)
    model = ResNet(ARCHS[a.arch]["widths"], ARCHS[a.arch]["blocks"], a.drop).to(DEV)
    if DEV == "cuda":
        model = model.to(memory_format=torch.channels_last)                   # faster convs on tensor cores
    P = n_params(model)
    assert P <= 5_000_000 or a.arch == "r18w", P

    teachers = []
    for spec in filter(None, a.teacher.split(",")):
        arch, path = spec.split(":", 1)
        t = ResNet(**ARCHS[arch]).to(DEV)
        t.load_state_dict(torch.load(path, map_location=DEV))
        t.eval().requires_grad_(False)
        teachers.append(t.to(memory_format=torch.channels_last) if DEV == "cuda" else t)
        print(f"teacher {arch} from {path}", flush=True)
    print(f"{name}: {P:,} trainable params on {DEV}", flush=True)

    xtr, ytr, _ = load("train")
    xte, _, ids = load("test")
    perm = np.random.default_rng(0).permutation(len(xtr))       # same split for every run
    tr, va = (perm, perm[:0]) if a.full else (perm[5000:], perm[:5000])
    if a.limit:
        tr, va = tr[:a.limit], va[:a.limit // 4]
    to_dev = lambda arr: torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(DEV)
    Xtr, Ytr = to_dev(xtr[tr]), torch.from_numpy(ytr[tr]).to(DEV)
    Xva, Yva = to_dev(xtr[va]), torch.from_numpy(ytr[va]).to(DEV)
    Xte = to_dev(xte)
    mean = Xtr.float().mean((0, 2, 3), keepdim=True) / 255
    std = Xtr.float().std((0, 2, 3), keepdim=True) / 255
    norm = lambda xb: (xb.float() / 255 - mean) / std

    decay = [p for p in model.parameters() if p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.ndim <= 1]                 # BN and bias: no weight decay
    opt = torch.optim.SGD([{"params": decay, "weight_decay": a.wd}, {"params": no_decay, "weight_decay": 0}],
                          lr=a.lr, momentum=0.9, nesterov=True)
    steps_per_ep = len(Xtr) // a.bs
    total = a.epochs * steps_per_ep
    warm = max(1, int(a.warmup * total))
    lr_at = lambda t: a.lr * t / warm if t < warm else a.lr * 0.5 * (1 + math.cos(math.pi * (t - warm) / (total - warm)))

    @torch.no_grad()
    def predict(X, tta=2):
        """Logits averaged over test-time views: the image and its flip, optionally at 4 extra 2-pixel shifts."""
        model.eval()
        shifts = [(0, 0)] if tta == 2 else [(0, 0), (2, 0), (-2, 0), (0, 2), (0, -2)]
        out = []
        for i in range(0, len(X), 1000):
            xp = F.pad(norm(X[i:i + 1000]), (2, 2, 2, 2), mode="reflect")
            acc = 0
            for dy, dx in shifts:
                xb = xp[:, :, 2 + dy:34 + dy, 2 + dx:34 + dx]
                with torch.autocast(DEV, dtype=torch.float16, enabled=amp):
                    acc = acc + model(xb).float() + model(xb.flip(3)).float()
            out.append(acc / (2 * len(shifts)))
        model.train()
        return torch.cat(out)

    hist, t0, step = [], time.time(), 0
    for ep in range(a.epochs):
        idx = torch.randperm(len(Xtr), device=DEV)
        tl, tc = 0.0, 0
        for b in range(steps_per_ep):
            j = idx[b * a.bs:(b + 1) * a.bs]
            x01 = Xtr[j].float() / 255
            if a.ta:
                x01 = trivial_aug(x01)
            xb, yb = augment((x01 - mean) / std, cutout=a.cutout), Ytr[j]
            lam, yb2 = 1.0, yb
            if a.cutmix and np.random.rand() < a.cutmix:                      # paste a box from another image
                lam = np.random.beta(1.0, 1.0)
                rp = torch.randperm(len(xb), device=DEV)
                h = int(32 * math.sqrt(1 - lam))
                cy, cx = np.random.randint(32), np.random.randint(32)
                y1, y2, x1, x2 = max(cy - h // 2, 0), min(cy + h // 2, 32), max(cx - h // 2, 0), min(cx + h // 2, 32)
                xb[:, :, y1:y2, x1:x2] = xb[rp, :, y1:y2, x1:x2]
                lam, yb2 = 1 - (y2 - y1) * (x2 - x1) / 1024, yb[rp]
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            if teachers:
                with torch.no_grad(), torch.autocast(DEV, dtype=torch.float16, enabled=amp):
                    t_logits = sum(t(xb).float() for t in teachers) / len(teachers)

            def loss_fn():
                with torch.autocast(DEV, dtype=torch.float16, enabled=amp):
                    logits = model(xb)
                    loss = lam * F.cross_entropy(logits, yb, label_smoothing=a.ls) + \
                        (1 - lam) * F.cross_entropy(logits, yb2, label_smoothing=a.ls)
                    if teachers:                                              # match the teachers' soft labels
                        kd = F.kl_div(F.log_softmax(logits.float() / a.kd_t, 1), F.softmax(t_logits / a.kd_t, 1),
                                      reduction="batchmean") * a.kd_t ** 2
                        loss = (1 - a.kd_a) * loss + a.kd_a * kd
                return loss

            opt.zero_grad(set_to_none=True)
            loss = loss_fn()
            scaler.scale(loss).backward()
            if a.sam:                       # SAM: move the weights rho along the gradient, take the gradient there
                params = [p for p in model.parameters() if p.grad is not None]
                inv = 1.0 / scaler.get_scale()                                # grads are still scaled
                gn = torch.norm(torch.stack([p.grad.norm() for p in params])) * inv
                if torch.isfinite(gn):                                        # on overflow the scaler skips the step
                    with torch.no_grad():
                        eps = [p.grad * (inv * a.sam / (gn + 1e-12)) for p in params]
                        torch._foreach_add_(params, eps)
                    opt.zero_grad(set_to_none=True)
                    scaler.scale(loss_fn()).backward()
                    with torch.no_grad():
                        torch._foreach_sub_(params, eps)
            scaler.step(opt)
            scaler.update()
            step += 1
            if b % 20 == 0:
                tl += loss.item(); tc += 1
        va_acc = (predict(Xva).argmax(1) == Yva).float().mean().item() if len(va) else float("nan")
        hist.append({"epoch": ep + 1, "train_loss": tl / tc, "val_acc": va_acc, "time_s": time.time() - t0})
        print(f"ep {ep + 1:3d}/{a.epochs}  loss {tl / tc:.3f}  val {100 * va_acc:.2f}%  {time.time() - t0:.0f}s", flush=True)

    final = {}
    if len(va):
        for k in (2, 10):
            final[f"val_acc_tta{k}"] = (predict(Xva, k).argmax(1) == Yva).float().mean().item()
        print("final val:", {k: f"{100 * v:.2f}%" for k, v in final.items()}, flush=True)
    te_logits = predict(Xte, a.tta).cpu()
    pred = te_logits.argmax(1).numpy()
    sub = os.path.join(RUNS, f"submission_{name}.csv")
    with open(sub, "w") as f:
        f.write("id,label\n" + "".join(f"{i},{p}\n" for i, p in zip(ids, pred)))
    torch.save(model.state_dict(), os.path.join(RUNS, f"{name}.pt"))
    np.save(os.path.join(RUNS, f"{name}_test_logits.npy"), te_logits.numpy())
    json.dump({"name": name, "args": vars(a), "params": P, "device": DEV, "history": hist, "final": final,
               "test_pred_counts": np.bincount(pred, minlength=10).tolist()},
              open(os.path.join(RUNS, f"{name}.json"), "w"), indent=1)
    print(f"wrote {sub}  test class counts {np.bincount(pred, minlength=10).tolist()}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import glob, subprocess, torch
assert torch.cuda.device_count() >= 1, 'No GPU: set the accelerator to GPU T4 x2'
assert glob.glob('/kaggle/input/**/train_labels.csv', recursive=True), 'Competition data not attached'
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python train.py --arch s4 --epochs 300 --ta --tta 10 --sam 0.05 --full --seed 1 > seed1.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python train.py --arch s4 --epochs 300 --ta --tta 10 --sam 0.05 --full --seed 2 > seed2.log 2>&1 &
wait
tail -n 4 seed1.log seed2.log
cp runs/submission_s4_ta_sam0.05_e300_full_s1.csv submission.csv
cp runs/submission_s4_ta_sam0.05_e300_full_s2.csv submission_s2.csv
ls -la *.csv